<a href="https://colab.research.google.com/github/SyhmZlkrn/ToneHound/blob/main/notebooks/ToneHound_Colab.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# ToneHound • MERT training in Colab

Code lives in GitHub. Your reviewed NAM/DI collection and experiment outputs live in your Google Drive. Select **Runtime → Change runtime type → GPU**, then run the cells in order. No TONE3000 API key or GitHub token is needed for this public code repository.

[Full walkthrough](https://github.com/SyhmZlkrn/ToneHound/blob/main/docs/CLOUD_TRAINING.md)


In [1]:
from pathlib import Path
import subprocess, sys
from google.colab import drive
drive.mount("/content/drive")
WORK = Path("/content")
ML_ROOT = Path("/content/drive/MyDrive/ToneHound-ML")
OUTPUT_ROOT = ML_ROOT / "experiments"
REPO = WORK / "ToneHound"


Mounted at /content/drive


## Get the training code

`CODE_REF` can be a reviewed commit hash for repeatable experiments. The run report records the resolved commit. Dependencies keep the runtime's existing compatible PyTorch/CUDA build. If Colab asks for a runtime restart after installation, restart and rerun the setup cells.


In [6]:
CODE_REF = "3e7857cdb78ce140953b54fa7c23db5a52973b40"
REMOTE = "https://github.com/SyhmZlkrn/ToneHound.git"
if not (REPO / ".git").exists():
    subprocess.run(["git", "clone", REMOTE, str(REPO)], check=True)
else:
    assert subprocess.check_output(["git", "-C", str(REPO), "remote", "get-url", "origin"], text=True).strip() == REMOTE
subprocess.run(["git", "-C", str(REPO), "fetch", "origin", CODE_REF], check=True)
subprocess.run(["git", "-C", str(REPO), "checkout", "--detach", "FETCH_HEAD"], check=True)
subprocess.run([sys.executable, "-m", "pip", "install", "-r", str(REPO / "requirements-training.txt")], check=True)


CompletedProcess(args=['/usr/bin/python3', '-m', 'pip', 'install', '-r', '/content/ToneHound/requirements-training.txt'], returncode=0)

In [7]:
import torch
assert sys.version_info >= (3, 10), "Use Python 3.10 or newer"
assert torch.cuda.is_available(), "Select a GPU runtime, reconnect, then rerun setup"
print("PyTorch:", torch.__version__, "| GPU:", torch.cuda.get_device_name())
print("Code:", subprocess.check_output(["git", "-C", str(REPO), "rev-parse", "HEAD"], text=True).strip())


PyTorch: 2.11.0+cu128 | GPU: Tesla T4
Code: 3e7857cdb78ce140953b54fa7c23db5a52973b40


In [3]:
from google.colab import drive
drive.mount('/content/drive')

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


## Unpack your private pilot and build the NAM A2 renderer

Upload `ToneHound-pilot-source.zip` (about 92 MB for the prepared pilot) into MyDrive first. This cell extracts it into `MyDrive/ToneHound-ML` and records completion, so an interrupted extraction can be rerun. If you already arranged the source folder manually, skip the unpack command. Building the pinned renderer uses CPU and does not require JUCE.


In [8]:
SOURCE_ZIP = ML_ROOT.parent / "ToneHound-pilot-source.zip"
subprocess.run([sys.executable, "engine/training/unpack_source.py", "--archive", str(SOURCE_ZIP), "--destination", str(ML_ROOT.parent)], cwd=REPO, check=True)
subprocess.run([sys.executable, "scripts/setup_training_renderer.py", "--jobs", "2"], cwd=REPO, check=True)
RENDERER = REPO / ".cache" / "training_renderer" / "nam-render"


## Choose a dataset and experiment

The prepared private pilot has 500 full rigs and 12 DI recordings. Its explicit manifest fixes the training/validation/test split. For a different collection, use `configs/dataset_manifest.example.json` and retain whole performance groups in one split.

`SMOKE_RUN` performs one optimizer step to check the setup; it still extracts the full frozen baseline. Use a NEW run name and set it to `False` for the full pilot. The source dataset is reusable across experiments.


In [9]:
CONFIG = "configs/mert_amp_v1.yaml"  # Later: mert_amp_lora.yaml / mert_amp_unfreeze_last.yaml
RUN_NAME = "mert_amp_v1"        # Use a NEW name for each config/dataset
SMOKE_RUN = False
SOURCE_MANIFEST = ML_ROOT / "source" / "manifest.json"
DATASET = ML_ROOT / "datasets" / "amp-tone-v1"
RUN = OUTPUT_ROOT / RUN_NAME


## Prepare the audio (skip these two cells for an attached, prepared dataset)

`--plan` checks local files, full-rig declarations, duplicate recordings, split isolation and estimated audio size. It does not download captures. The next cell renders the dataset and can resume after interruption.


In [10]:
subprocess.run([sys.executable, "engine/training/build_dataset.py", "--manifest", str(SOURCE_MANIFEST), "--output", str(DATASET), "--plan"], cwd=REPO, check=True)


CompletedProcess(args=['/usr/bin/python3', 'engine/training/build_dataset.py', '--manifest', '/content/drive/MyDrive/ToneHound-ML/source/manifest.json', '--output', '/content/drive/MyDrive/ToneHound-ML/datasets/amp-tone-v1', '--plan'], returncode=0)

In [ ]:
subprocess.run([sys.executable, "engine/training/build_dataset.py", "--manifest", str(SOURCE_MANIFEST), "--output", str(DATASET), "--renderer", str(RENDERER)], cwd=REPO, check=True)


## Train, or resume the same run

This saves to persistent output storage. A frozen baseline is fitted first, using only training and validation data. LoRA and partial fine-tuning update MERT; the frozen configuration updates only the small retrieval head. The active desktop matcher is never changed.


In [ ]:
import yaml
cfg = yaml.safe_load((REPO / CONFIG).read_text())
if SMOKE_RUN:
    cfg["training"].update(epochs=1, steps_per_epoch=1, classes_per_batch=2, eval_batch_size=1, checkpoint_every_steps=1)
run_config = WORK / "tonehound-run-config.yaml"
run_config.write_text(yaml.safe_dump(cfg, sort_keys=False))
command = [sys.executable, "engine/training/train_mert.py", "--config", str(run_config), "--dataset", str(DATASET), "--output", str(RUN)]
if (RUN / "checkpoint" / "last.pt").exists():
    command.append("--resume")
subprocess.run(command, cwd=REPO, check=True)


## Inspect validation before opening the test set

For a real comparison, complete the planned experiment configurations on the **same dataset**, then select using validation only. Do not repeatedly change settings after looking at the test score. The example selection command is in `docs/CLOUD_TRAINING.md`.


In [ ]:
import json
print(json.dumps(json.loads((RUN / "metrics.json").read_text()), indent=2))


## Final held-out evaluation (run only after selecting the experiment)

A separate cell makes test access deliberate. Compare against the frozen encoder/projection on the same dataset. When the selected run uses MERT-95M, set `REFERENCE_RUN` to the completed `mert_amp_v1` experiment to also compare with the current 330M encoder's projection method. Old 51-capture percentages are not comparable to a new 500-capture test.


In [ ]:
REFERENCE_RUN = None  # e.g. OUTPUT_ROOT / "mert_amp_v1"
command = [sys.executable, "engine/training/evaluate_mert.py", "--run", str(RUN), "--dataset", str(DATASET)]
if REFERENCE_RUN is not None:
    command += ["--reference-run", str(REFERENCE_RUN)]
subprocess.run(command, cwd=REPO, check=True)


## Collect the results

Keep the **whole experiment folder**: config, checkpoint, metrics, training history, retrieval test, confusion counts and run information. `checkpoint/best.safetensors` contains the head and any trained MERT updates; it requires the pinned base model and is not a drop-in replacement for the desktop app's `active.npz`. Bring this folder back for local evaluation/integration. Keep audio, checkpoints, API keys and notebook outputs out of the source repository.


In [ ]:
for path in sorted(RUN.rglob("*")):
    if path.is_file():
        print(path.relative_to(RUN), f"{path.stat().st_size / 1024:.1f} KiB")
